# PyPSA-Earth Clustering Validation

Runs the model at several `{clusters}` levels (default k-means/Voronoi clustering) and, optionally, one administrative (GADM-boundary) clustering case, then compares key system outcomes (cost, RE share, CO2, line congestion) against a chosen reference resolution.

**Two run modes**, set via `RUN_SNAKEMAKE` in the configuration cell:

- `True`: this notebook triggers PyPSA-Earth's Snakemake workflow for each case, then analyzes the results.
- `False`: assumes you already solved the networks yourself, and the notebook only loads and analyzes them.

**Administrative clustering note**: this is a global config switch in PyPSA-Earth (`clustering.alternative_clustering: true`, or in newer versions `clustering.mode: administrative` with `scenario.clusters: ["adm"]` -- check `config.default.yaml` in your installed version for the exact keys), not a per-target wildcard. The cleanest way to run it alongside the k-means sweep is a second config file with that switch enabled, pointed to by `ADMIN_CONFIGFILE`.

Before running, edit the **Configuration** cell to match your repo layout and your own solved-network file naming (depends on your `config.yaml` scenario settings for `{opts}`, `{ll}`, `{ec}`, which PyPSA-Earth doesn't fix generically).

In [ ]:
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pypsa
from IPython.display import display

%matplotlib inline

## Configuration

Edit these values for your setup.

In [ ]:
# Cluster counts to test with the default (k-means/Voronoi) clustering.
# Include your intended operating resolution and, if feasible, a
# high-resolution case to serve as the reference.
CLUSTER_LEVELS = [4, 10, 20, 50, 100]

# Cluster level (from CLUSTER_LEVELS) to treat as "ground truth" for the
# percent-deviation columns. Leave as None to use the finest level instead.
REFERENCE_LEVEL = None

# Whether to also run/include an administrative (GADM-boundary) clustering
# case alongside the k-means sweep.
INCLUDE_ADMINISTRATIVE = False

# Wildcard value substituted into NETWORK_TEMPLATE's {clusters} slot for the
# administrative case. Depends on your PyPSA-Earth version: some versions use
# the literal string "adm", others still expect a plain number even when
# alternative_clustering is enabled. Check your version's wildcard docs.
ADMIN_CLUSTER_WILDCARD = "adm"

# Label used in the results table and plots for the administrative case.
ADMIN_LABEL = "Administrative (GADM)"

# Path to a separate PyPSA-Earth config file that has administrative
# clustering enabled (e.g. a copy of config.yaml with
# clustering.alternative_clustering: true, or clustering.mode: administrative
# + scenario.clusters: ["adm"], depending on your version). Only used when
# RUN_SNAKEMAKE is True.
ADMIN_CONFIGFILE = Path("configs/config.administrative.yaml")

# Path to the pypsa-earth repository root (only used if RUN_SNAKEMAKE is True).
PYPSA_EARTH_DIR = Path("pypsa-earth")

# Whether this notebook should invoke Snakemake itself, or just analyze
# networks you have already solved.
RUN_SNAKEMAKE = True
SNAKEMAKE_CORES = 4

# Filename template for a SOLVED network, relative to PYPSA_EARTH_DIR.
# {clusters} is filled in per case. Check your own results/networks/ folder
# after one manual run to get the exact pattern (it depends on your
# scenario's opts/ll/ec wildcards, e.g. "Co2L-24h", "v-opt", etc.).
NETWORK_TEMPLATE = "results/networks/elec_s_{clusters}_ec_lcopt_Co2L-1h.nc"

# Carriers treated as renewables for the RE-share calculation. This matches
# PyPSA-Earth's own default electricity.renewable_carriers list. Note that
# "hydro" (reservoir) and "PHS" (pumped storage) are StorageUnit components in
# PyPSA-Earth, not Generator components -- only "ror" (run-of-river) lives in
# n.generators. dispatch_by_carrier() below accounts for this by pulling
# dispatch from both n.generators_t.p and n.storage_units_t.p, so reservoir
# hydro is correctly counted even though it isn't a Generator carrier.
RE_CARRIERS = ["solar", "onwind", "offwind-ac", "offwind-dc", "hydro", "ror"]

OUTPUT_DIR = Path("clustering_validation")
OUTPUT_DIR.mkdir(exist_ok=True)

## Case builder and Snakemake runner

In [ ]:
def build_cases() -> list[dict]:
    """Assemble the list of clustering cases to run/analyze."""
    cases = [
        {
            "wildcard": str(level),
            "label": str(level),
            "configfile": None,
            "is_admin": False,
        }
        for level in CLUSTER_LEVELS
    ]
    if INCLUDE_ADMINISTRATIVE:
        cases.append(
            {
                "wildcard": ADMIN_CLUSTER_WILDCARD,
                "label": ADMIN_LABEL,
                "configfile": ADMIN_CONFIGFILE,
                "is_admin": True,
            }
        )
    return cases


def network_path_for(wildcard: str) -> Path:
    return PYPSA_EARTH_DIR / NETWORK_TEMPLATE.format(clusters=wildcard)


def run_snakemake_target(wildcard: str, configfile: Path = None) -> Path:
    """Trigger the PyPSA-Earth workflow up to a solved network for one case.

    If `configfile` is given, it is passed via --configfile so this case runs
    with a different config (e.g. administrative clustering enabled) without
    touching the config used for the rest of the sweep.
    """
    target = NETWORK_TEMPLATE.format(clusters=wildcard)
    cmd = ["snakemake", "-j", str(SNAKEMAKE_CORES)]
    if configfile is not None:
        cmd += ["--configfile", str(configfile)]
    cmd.append(target)
    print(f"[{wildcard}] running: {' '.join(cmd)}")
    result = subprocess.run(cmd, cwd=PYPSA_EARTH_DIR)
    if result.returncode != 0:
        raise RuntimeError(f"Snakemake failed for case {wildcard}")
    return network_path_for(wildcard)

## KPI extraction

Loads a solved network and computes the metrics used for the convergence comparison: total system cost, RE share (correctly including reservoir hydro, which PyPSA-Earth models as a `StorageUnit`, not a `Generator`), CO2 emissions, and line loading.

In [ ]:
def line_loading_stats(n: pypsa.Network):
    """Mean and max line loading as a fraction of thermal capacity."""
    if n.lines_t.p0.empty:
        return None, None
    s_nom = n.lines.s_nom_opt.where(n.lines.s_nom_opt > 0, n.lines.s_nom)
    loading = n.lines_t.p0.abs().div(s_nom.replace(0, pd.NA), axis=1)
    stacked = loading.stack(dropna=True)
    if stacked.empty:
        return None, None
    return float(stacked.mean()), float(stacked.max())


def co2_emissions(n: pypsa.Network):
    """Total CO2 emissions from generator dispatch, using carrier emission factors.

    Assumes add_electricity populated n.carriers.co2_emissions and
    n.generators.efficiency, as PyPSA-Earth does by default. Adjust if your
    setup stores emissions differently (e.g. sector-coupled models).
    """
    if "co2_emissions" not in n.carriers.columns:
        return None
    gens = n.generators
    ef = gens.carrier.map(n.carriers.co2_emissions).fillna(0)
    eff = gens.efficiency.replace(0, pd.NA)
    weights = n.snapshot_weightings.generators
    dispatch = n.generators_t.p.mul(weights, axis=0)
    emissions = (dispatch.div(eff, axis=1).fillna(0) * ef).sum().sum()
    return float(emissions)


def dispatch_by_carrier(n: pypsa.Network) -> pd.Series:
    """Total dispatched energy per carrier, combining Generators and StorageUnits.

    Reservoir hydro ("hydro") and pumped storage ("PHS") are StorageUnit
    components in PyPSA-Earth, so their dispatch would be missed if only
    n.generators were scanned. Storage charging (negative p) is excluded --
    only discharge to the grid counts as generation.
    """
    parts = [n.generators_t.p.sum().groupby(n.generators.carrier).sum()]
    if not n.storage_units_t.p.empty:
        discharge = n.storage_units_t.p.clip(lower=0)
        parts.append(discharge.sum().groupby(n.storage_units.carrier).sum())
    return pd.concat(parts).groupby(level=0).sum()


def extract_kpis(network_path: Path, case: dict) -> dict:
    n = pypsa.Network(str(network_path))

    gen_by_carrier = dispatch_by_carrier(n)
    total_gen = gen_by_carrier.sum()
    re_gen = gen_by_carrier[gen_by_carrier.index.isin(RE_CARRIERS)].sum()
    re_share = float(re_gen / total_gen) if total_gen > 0 else None

    mean_loading, max_loading = line_loading_stats(n)

    return {
        "label": case["label"],
        "wildcard": case["wildcard"],
        "is_admin": case["is_admin"],
        # Actual node count read from the solved network, rather than the
        # requested wildcard, so administrative (and any other non-numeric)
        # cases land at their true resolution on the x-axis.
        "n_nodes": len(n.buses),
        "total_cost": float(n.objective),
        "re_share": re_share,
        "co2_emissions": co2_emissions(n),
        "mean_line_loading": mean_loading,
        "max_line_loading": max_loading,
        "capacity_by_carrier": n.generators.groupby("carrier").p_nom_opt.sum().to_dict(),
    }

## Run the sweep

For each case: run Snakemake (if `RUN_SNAKEMAKE`) or locate the already-solved network, then extract KPIs.

In [ ]:
records = []

for case in build_cases():
    path = (
        run_snakemake_target(case["wildcard"], case["configfile"])
        if RUN_SNAKEMAKE
        else network_path_for(case["wildcard"])
    )
    if not path.exists():
        print(f"[{case['wildcard']}] missing solved network at {path}, skipping.")
        continue
    records.append(extract_kpis(path, case))

if not records:
    raise SystemExit("No solved networks found. Check NETWORK_TEMPLATE and paths.")

print(f"Collected KPIs for {len(records)} case(s).")

## Build the results table

Computes percent deviation of each case from the chosen reference resolution and writes the results to CSV.

In [ ]:
df = pd.DataFrame(records).sort_values("n_nodes").reset_index(drop=True)

regular = df[~df["is_admin"]]
if REFERENCE_LEVEL is not None:
    ref_rows = regular[regular["wildcard"] == str(REFERENCE_LEVEL)]
    reference_row = ref_rows.iloc[0] if not ref_rows.empty else regular.loc[regular["n_nodes"].idxmax()]
else:
    reference_row = regular.loc[regular["n_nodes"].idxmax()]

for col in ["total_cost", "re_share", "co2_emissions", "mean_line_loading"]:
    ref_value = reference_row.get(col)
    if ref_value in (None, 0) or pd.isna(ref_value):
        continue
    df[f"{col}_pct_dev"] = (df[col] - ref_value) / ref_value * 100

csv_path = OUTPUT_DIR / "clustering_validation_results.csv"
df.to_csv(csv_path, index=False)

print(f"Reference case: {reference_row['label']} ({int(reference_row['n_nodes'])} nodes)")
print(f"Results written to: {csv_path}")
display(df.drop(columns=["capacity_by_carrier"]))

## Convergence plots

One panel per KPI vs. number of nodes. The administrative case (if included) is overlaid as a separate marker since it isn't part of the same continuous sweep. Font sizes are set to stay readable when the figure is placed in a PDF/thesis at report scale. Saved as both PNG and PDF.

In [ ]:
def plot_convergence(df: pd.DataFrame, reference_n_nodes: float):
    """Readable, PDF-ready convergence plots: one panel per KPI vs. node count."""
    kpi_labels = {
        "total_cost": "Total system cost",
        "re_share": "RE share of generation",
        "co2_emissions": "CO2 emissions",
        "mean_line_loading": "Mean line loading (fraction of capacity)",
    }

    regular = df[~df["is_admin"]].sort_values("n_nodes")
    admin = df[df["is_admin"]]

    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    axes = axes.flatten()

    for ax, (col, label) in zip(axes, kpi_labels.items()):
        if col not in df.columns:
            continue
        ax.plot(
            regular["n_nodes"], regular[col],
            marker="o", linewidth=2, markersize=7, label="k-means sweep",
        )
        ax.axvline(reference_n_nodes, color="grey", linestyle="dotted", linewidth=1.5)
        if not admin.empty:
            ax.scatter(
                admin["n_nodes"], admin[col],
                marker="*", s=180, color="firebrick", zorder=5,
                label=ADMIN_LABEL,
            )
        ax.set_xlabel("Number of nodes", fontsize=12)
        ax.set_ylabel(label, fontsize=12)
        ax.set_title(label, fontsize=13)
        ax.tick_params(labelsize=10)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=9)

    fig.suptitle("PyPSA-Earth clustering resolution convergence", fontsize=15)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(OUTPUT_DIR / "clustering_convergence.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / "clustering_convergence.pdf", bbox_inches="tight")
    return fig


fig = plot_convergence(df, reference_row["n_nodes"])
plt.show()